# Week 4 — Deeper OOP for Agent Design

Covers: abstract base classes (agent contracts), composition vs. inheritance, and `dataclasses`.

## 1. Abstract base classes — defining a common agent contract

In [ ]:
from abc import ABC, abstractmethod

class Agent(ABC):
    """Every agent role in a multi-agent system must implement `act()`.
    ABC + @abstractmethod makes this a compile-time-checked contract, not a convention
    someone can silently forget."""

    def __init__(self, name):
        self.name = name

    @abstractmethod
    def act(self, task: str) -> str:
        ...

class ClassifierAgent(Agent):
    def act(self, task: str) -> str:
        return f"[{self.name}] classified: '{task}' -> billing_issue"

class DraftingAgent(Agent):
    def act(self, task: str) -> str:
        return f"[{self.name}] drafted a response for: '{task}'"

agents = [ClassifierAgent("Classifier"), DraftingAgent("Drafter")]
for agent in agents:
    print(agent.act("Customer was charged twice for the same order"))

In [ ]:
# Trying to instantiate the abstract base directly is a hard error — that's the point.
try:
    broken = Agent("NoImplementation")
except TypeError as e:
    print("As expected, this fails:", e)

## 2. Composition vs. inheritance — "has a tool" vs. "is a role"

In [ ]:
# Composition: an agent HAS a tool it can call. This is the more flexible, more common
# pattern in real multi-agent frameworks — tools are swappable without changing the class hierarchy.

class DatabaseTool:
    def query(self, sql):
        return f"(mock) ran query: {sql}"

class WebSearchTool:
    def search(self, query):
        return f"(mock) search results for: {query}"

class ResearchAgent(Agent):
    def __init__(self, name, tools: dict):
        super().__init__(name)
        self.tools = tools   # composition: the agent HOLDS its tools, doesn't inherit them

    def act(self, task: str) -> str:
        db_result = self.tools["db"].query("SELECT * FROM orders WHERE flagged = 1")
        web_result = self.tools["web"].search(task)
        return f"[{self.name}] combined: {db_result} | {web_result}"

researcher = ResearchAgent("Researcher", tools={"db": DatabaseTool(), "web": WebSearchTool()})
print(researcher.act("duplicate charge policy"))

## 3. `dataclasses` — lightweight structured state

In [ ]:
from dataclasses import dataclass, field
from typing import List

@dataclass
class AgentState:
    task: str
    history: List[str] = field(default_factory=list)   # mutable default done safely
    confidence: float = 0.0

    def log(self, entry: str):
        self.history.append(entry)

state = AgentState(task="Resolve duplicate charge")
state.log("classified as billing_issue")
state.log("drafted refund response")
state.confidence = 0.91

print(state)
print("History so far:", state.history)